In [ ]:
from bs4 import BeautifulSoup
import json

with open(r"walktrough.html", "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

Here, we investigate our options for finding the correct location of the information we want to extract

In [ ]:
#Getting to the correct place & extracting the correct info
westlim_list = soup.find("a", id="WestLimgrave")
section1 = westlim_list.find_parent("div", class_="col-sm-1")
title1 = section1.find("h3").get_text(" ",strip=True)
lvl_stat1 = section1.find("p").get_text(" ",strip=True)
# Getting to 2nd div is not straightforward
main1=westlim_list.find_parent("div", class_="row")
children_divs = [child for child in main1.find_all("div", recursive=False)]
index = children_divs.index(section1)
westlim_desc = children_divs[index + 1]
paragraphs = [p.get_text(" ", strip=True) for p in westlim_desc.find_all("p") if p.get_text(strip=True)]

In [3]:
#Inserting data to our dictionary
metadata = {}

if "Level" in lvl_stat1:
    metadata["level_range"]= lvl_stat1.split("Level :")[1].split("Upgrades")[0].strip()
if "Upgrades" in lvl_stat1:
    metadata["upgrade_range"]= lvl_stat1.split("Upgrades :")[1].strip()

stepsreg1=[]
stepdad1 = section1.find("ol")

for i, li in enumerate(stepdad1.find_all("li"),start=1):
    stepsreg1.append({
        "step": i,
        "text": li.get_text(" ", strip=True)
    })

datareg1= {
    "region": title1,
    "Stats" : lvl_stat1,
    "Steps" : stepsreg1,
    "Step Desc" : paragraphs
}

Now, we have to do this for all regions via a loop. Let's get a list of the regions.

In [ ]:
list1=soup.find("div",class_="col-sm-3")
parent1=list1.find_parent("div",class_="row")
regions=[li.get_text(" ", strip=True).replace(" ↵","") for li in parent1.find_all("li")]
print(regions)

Data in the HTML may not be the cleanest and the most organized. For instance, the title of the regions may not be the same with their ids, or there might be special cases where some info is inserted into the HTML in a different fashion than the rest.

Let's also we extract the ids. However, the way we extract them already captures the columns(classes) we want to reach and more. Thus, going after id's for extracting data seems redundant.

In [ ]:
allid=soup.find_all("div",class_="col-sm-1")
idlist=[]

for i in range(1,len(allid)):
    idreg=allid[i].find("a",id=True)
    idlist.append(idreg["i" \
    "" \
    "" \
    "d"])
print(idlist)

It appears that all region walktrough paragraphs are located under a `col-sm-8` except one! In this case, the convenient choice would be reaching all paragraphs via `col-sm-8`. To do so, we have to first add the missing paragraph under the same class.

In [ ]:
reg_info1=soup.find_all("div",class_="col-sm-1")
#:getting missing paragraph
missingtext=reg_info1[10].find_parent("div",class_="row").find_all("p")
textcol=[]
for i in range(1,len(missingtext)):
    textcol.append(missingtext[i].get_text())
missing_txt="".join(textcol)
missing_reg=soup.new_tag("p")
missing_reg.string="".join(missing_txt)
shell=soup.new_tag("div")
shell.append(soup.new_tag("p"))
shell.append(missing_reg)
#:adding missing paragraph
reg_info2=soup.find_all("div",class_="col-sm-2")
reg_info2.insert(10,shell)

Now, we have access to all data in a systematic way. We'll write a loop to fill out our `datareg` dictionary.

In [ ]:
bigdata=[]
#: Labeling relevant info
for i in range(1,len(reg_info1)):
    title=reg_info1[i].find("h3").get_text(" ",strip=True)

    if reg_info1[i].find("p") is not None:
        lvl_stat= reg_info1[i].find("p").get_text(" ",strip=True)
    else:
        lvl_stat=" "

    #: Steps to Follow
    stepsreg=[]
    steps_main=reg_info1[i].find("ol")

    for n, li in enumerate(steps_main.find_all("li"),start=1):
        stepsreg.append({
            "step" : n,
            "text" : li.get_text(" ",strip=True)
        })
    #: Region Walktrough --> This loop gets all the text in stephelp while preserving the features i.e plain text or a list.
    allpar=reg_info2[i].find_all(["p","ul","ol"],recursive=False)
    par=[]
    for m in range(1,len(allpar)):
        if allpar[m].name == "p":
            par.append(allpar[m].get_text(separator=" ",strip=True).replace( "<Unwanted strings>",""))
        elif allpar[m].name == "ul":
            lis=[]
            k=1
            for li in allpar[m].find_all("li"):
                lis.append(str(k)+"-"+li.get_text(" ", strip=True).replace( "<Unwanted strings>",""))
                k+=1
            par.append(lis)
        elif allpar[m].name == "ol":
            lis=[]
            k=1
            for li in allpar[m].find_all("li"):
                lis.append(str(k)+"-"+li.get_text(" ", strip=True).replace( "<Unwanted strings>",""))
                k+=1
            par.append(lis)

    #: Database
    datareg= {
        "region" : title,
        "Stats" : lvl_stat,
        "Steps" : stepsreg,
        "Step Desc" : par
    }
    bigdata.append(datareg)

    with open("bigdata.json","w") as f:
        json.dump(bigdata,f)